# Multi-Agent RAG POC - Feature 2: Optimized Screening
This notebook verifies the Batched Screening Agent.

In [ ]:
import sys
import os
sys.path.append('..')

from dotenv import load_dotenv
load_dotenv('../.env')

from rag.ingest import load_pdf, chunk_text
from rag.embed import embed_texts
from rag.index import VectorStore
from orchestration.graph import app

### 1. Setup Data with Mix of Relevant & Irrelevant
To test screening, we need irrelevant documents.

In [ ]:
# Create dummy docs directly in memory for this test
relevant_text = "Machine unlearning allows deleting data from a trained model. It is crucial for privacy compliance like GDPR."
irrelevant_text = "The history of ancient Rome is marked by the transition from Republic to Empire. Caesar played a key role."
mixed_text = "Deep learning models require massive GPU resources for training. Optimization of hyperparameters is key."

chunks = [
    {"page_content": relevant_text, "metadata": {"id": 1, "type": "relevant"}},
    {"page_content": irrelevant_text, "metadata": {"id": 2, "type": "irrelevant"}},
    {"page_content": mixed_text, "metadata": {"id": 3, "type": "ambiguous"}}
]

chunk_texts = [c['page_content'] for c in chunks]
embeddings = embed_texts(chunk_texts)

vs = VectorStore(persist_directory="../data/chroma_db_screening_test")
# Clear previous test data if any (hacky way for POC)
try:
    vs.client.delete_collection("research_papers")
    vs.collection = vs.client.create_collection("research_papers")
except:
    pass

vs.add_documents(chunks, embeddings)
print("Indexed mixed documents")

### 2. Verify Screening Agent Isolated

In [ ]:
from agents.screening import ScreeningAgent

agent = ScreeningAgent()
query = "Machine Unlearning methods"

# Pass all chunks as if they were retrieved
screened = agent.screen(chunks, query, batch_size=3)

print(f"\nQuery: {query}")
print(f"Input count: {len(chunks)}")
print(f"Output count: {len(screened)}")
for doc in screened:
    print(f"KEPT: {doc['page_content'][:50]}...")

### 3. Run Full Pipeline with Screening

In [ ]:
# Override the retriever in the graph (conceptually) or just run the graph which uses the main DB
# For this test, let's just run the graph and see if it screens the production DB results
# Note: The graph uses 'Retriever()' which loads the DEFAULT 'data/chroma_db', not our test one above.
# So this part tests the Integration, but maybe not the exact chunks above unless we injected them.

initial_state = {
    "query": "Machine unlearning",
    "expanded_queries": [],
    "retrieved_docs": [],
    "final_response": ""
}

print("Running full pipeline...")
result = app.invoke(initial_state)

print("FINAL SYNTHESIS:")
print(result['final_response'])